# Cross-Dataset Evaluation

This notebook evaluates models across datasets to measure domain shift robustness:
- Train on RadioML2016 → Evaluate on RadioML2018
- Train on RadioML2018 → Evaluate on RadioML2016

Uses overlapping modulation classes for fair comparison.

In [ ]:
import sys
from pathlib import Path

src_path = Path("../src")
if src_path.exists():
    sys.path.insert(0, str(src_path.resolve()))

import matplotlib.pyplot as plt
import torch

from robust_amc.data import OVERLAPPING_CLASSES
from robust_amc.data.radioml_loader import MODULATION_CLASSES as CLASSES_2016
from robust_amc.data.radioml2018_loader import MODULATION_CLASSES_2018 as CLASSES_2018
from robust_amc.evaluation import (
    get_class_mapping,
    load_overlapping_data,
    evaluate_cross_dataset,
    plot_accuracy_vs_snr,
)
from robust_amc.models import create_pfcnn, create_clsr_amc
from robust_amc.utils import get_device

## 1. Setup

In [ ]:
DATA_PATH_2016 = Path("../data/RML2016.10a_dict.pkl")
DATA_PATH_2018 = Path("../data/GOLD_XYZ_OSC.0001_1024.hdf5")
CHECKPOINTS_DIR = Path("../checkpoints")

device = get_device("auto")
print(f"Using device: {device}")
print(f"\nOverlapping classes ({len(OVERLAPPING_CLASSES)}):")
for cls in OVERLAPPING_CLASSES:
    print(f"  - {cls}")

## 2. Configuration

Choose which direction to evaluate:

In [ ]:
# Configuration - change these to evaluate different scenarios
TRAIN_DATASET = "2016"  # Dataset the models were trained on
EVAL_DATASET = "2018"   # Dataset to evaluate on

print(f"Evaluating: Train on {TRAIN_DATASET} -> Eval on {EVAL_DATASET}")

## 3. Load Evaluation Data

In [ ]:
try:
    data, labels, snrs, class_names = load_overlapping_data(
        EVAL_DATASET,
        data_path_2016=DATA_PATH_2016,
        data_path_2018=DATA_PATH_2018,
        max_samples=50000,  # Limit for faster evaluation
    )
    print(f"Loaded {EVAL_DATASET} dataset:")
    print(f"  Samples: {len(data)}")
    print(f"  Classes: {len(class_names)}")
    print(f"  SNR range: {snrs.min():.0f} to {snrs.max():.0f} dB")
except FileNotFoundError as e:
    print(f"Error: {e}")

## 4. Load Models and Create Class Mapping

In [ ]:
# Get training classes based on training dataset
train_classes = CLASSES_2016 if TRAIN_DATASET == "2016" else CLASSES_2018
use_2018_mapping = TRAIN_DATASET == "2018"

# Build class mapping from training indices to overlapping indices
class_mapping = get_class_mapping(
    train_classes,
    OVERLAPPING_CLASSES,
    use_2018_mapping=use_2018_mapping,
)

print(f"Class mapping ({len(class_mapping)} classes):")
for train_idx, overlap_idx in sorted(class_mapping.items()):
    print(f"  {train_classes[train_idx]} -> {OVERLAPPING_CLASSES[overlap_idx]}")

In [ ]:
# Load models
models = {}

baseline_path = CHECKPOINTS_DIR / f"baseline_{TRAIN_DATASET}" / "best_model.pt"
if baseline_path.exists():
    model = create_pfcnn(num_classes=len(train_classes))
    ckpt = torch.load(baseline_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    models["Baseline"] = model
    print(f"Loaded: Baseline from {baseline_path}")

mda_path = CHECKPOINTS_DIR / f"mda_dmc_{TRAIN_DATASET}" / "best_model.pt"
if mda_path.exists():
    model = create_pfcnn(num_classes=len(train_classes))
    ckpt = torch.load(mda_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    models["MDA-DMC"] = model
    print(f"Loaded: MDA-DMC from {mda_path}")

clsr_path = CHECKPOINTS_DIR / f"clsr_amc_{TRAIN_DATASET}" / "best_model.pt"
if clsr_path.exists():
    model = create_clsr_amc(num_classes=len(train_classes))
    ckpt = torch.load(clsr_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    models["CLSR-AMC"] = model
    print(f"Loaded: CLSR-AMC from {clsr_path}")

if not models:
    print(f"No models found trained on {TRAIN_DATASET}!")

## 5. Cross-Dataset Evaluation

In [ ]:
results = {}

for name, model in models.items():
    print(f"\nEvaluating: {name}")
    result = evaluate_cross_dataset(
        model, data, labels, snrs, device,
        train_class_to_overlap=class_mapping,
    )
    results[name] = result
    print(f"  Overall accuracy: {result['accuracy']:.2%}")

## 6. Accuracy vs SNR

In [ ]:
if results:
    # Collect SNR values and accuracies for plotting
    snr_values = sorted(list(results[list(results.keys())[0]]["snr_accuracy"].keys()))
    
    snr_accuracies = {}
    for name, result in results.items():
        snr_accuracies[name] = [result["snr_accuracy"].get(snr, 0) for snr in snr_values]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    plot_accuracy_vs_snr(
        snr_values, snr_accuracies,
        title=f"Cross-Dataset: Train {TRAIN_DATASET} -> Eval {EVAL_DATASET}",
        ax=ax,
    )
    plt.show()

## 7. Summary

In [ ]:
if results:
    print(f"Cross-Dataset Evaluation: Train {TRAIN_DATASET} -> Eval {EVAL_DATASET}")
    print("=" * 50)
    print(f"{'Model':<15} {'Accuracy':>12}")
    print("-" * 29)
    for name, result in results.items():
        print(f"{name:<15} {result['accuracy']:>12.2%}")
    
    print("\nNote: This measures robustness to domain shift between datasets.")
    print("Lower accuracy compared to same-dataset evaluation indicates domain shift.")